In [1]:
import os
import sys

root = "/tmp2/maitanha/vgu/cll_vlm/cll_vlm"

if root not in sys.path:
    sys.path.append(root)

import torch
import re
import csv
import random
import numpy as np
import pandas as pd
import yaml
import json
from pathlib import Path
from collections import Counter
from IPython.display import display
from argparse import ArgumentParser
from torch.utils.data import DataLoader
from tqdm import tqdm
import pdb
from dataset.cifar10 import CIFAR10Dataset
from dataset.cifar20 import CIFAR20Dataset, CIFAR100Dataset
from dataset.tiny200 import Tiny200Dataset

from models.llava_classifier import LLaVAClassifier
from models.qwen_classifier import QWENClassifier
from models.clip_model import CLIPModel
from PIL import Image   

/tmp2/maitanha/vgu/cll_vlm/venv_cll_qwen/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python version is above 3.10, patching the collections module.


/tmp2/maitanha/vgu/cll_vlm/venv_cll_qwen/lib/python3.13/site-packages/transformers/models/auto/image_processing_auto.py:647: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(


In [2]:
def load_config(config_path):
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Config file '{config_path}' not found.")
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    return config

def collate_fn(batch):
    images, labels = zip(*batch)
    return list(images), list(labels)

def load_dataset(data_name):
    DATA_DIR = "/tmp2/maitanha/vgu/cll_vlm/cll_vlm/data"
    data_root_path = os.path.join(DATA_DIR, data_name)

    if data_name == "cifar10":
        dataset = CIFAR10Dataset(
            root=data_root_path,
            train=True,
            transform=None
        )
    elif data_name == "cifar20":
        dataset = CIFAR20Dataset(
            root=data_root_path,
            train=True,
            transform=None
        )
    elif data_name == "cifar100":
        dataset = CIFAR100Dataset(
            root = data_root_path,
            train=True,
            transform=None
        )
        fine_classes_raw = dataset.get_fine_classes()
        fine_classes = [
            CIFAR100Dataset.preprocess_label(lbl)
            for lbl in fine_classes_raw
        ]
        coarse_classes = dataset.get_coarse_classes()
    elif data_name == "tiny200":
        dataset = Tiny200Dataset(
            root=data_root_path,
            train=True,
            transform=None
        )
    else:
        raise ValueError(f"Dataset '{data_name}' chưa được hỗ trợ trong hàm load_dataset.")
    
    return dataset

In [3]:
dataset = load_dataset("cifar100")
fine_classes_raw = dataset.get_fine_classes()
fine_classes = [
    CIFAR100Dataset.preprocess_label(lbl)
    for lbl in fine_classes_raw
]
coarse_classes = dataset.get_coarse_classes()

original_dataset, shuffled_dataset = dataset.get_shuffled_labels_dataset(seed=42)

# Analyze Baseline 3.2

In [1]:
def analyze_result(json_path):
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        return {
            "error": str(e),
            "file": Path(json_path).name
        }

    total = len(data)
    correct_candidate = 0
    correct_final = 0
    empty_answer = 0
    
    for item in data:
        true_label = str(item.get('true_label', '')).strip().lower()
        candidate = item.get('candidate_answer', [])
        answer = item.get('clip_answer', [])

        if isinstance(candidate, str): 
            candidate = [candidate]
        elif candidate is None:
            candidate = []
        else:
            candidate = [str(c).strip().lower() for c in candidate]

        if isinstance(answer, str): answer = [answer]
        candidate = [str(c).strip().lower() for c in candidate]
        answer = [str(a).strip().lower() for a in answer]

        if not answer or (len(answer) == 1 and answer[0] == ""): # None or empty
            empty_answer += 1
            continue

        if true_label in candidate:
            correct_candidate += 1
        if true_label in answer:
            correct_final += 1

    return {
        'file': Path(json_path).name,
        'total': total,
        'c_count': correct_candidate,
        'f_count': correct_final,
        'candidate_acc': (correct_candidate / total) * 100 if total > 0 else 0,
        'final_acc': (correct_final / total) * 100 if total > 0 else 0,
        'missed': (correct_candidate - correct_final),
        'NO/Empty ans': empty_answer,
        'error': None
    }

from pathlib import Path
import pandas as pd
import json

# --- THỰC THI VÀ HIỂN THỊ ---
JSON_DIR = Path("/tmp2/maitanha/vgu/cll_vlm/cll_vlm/results/baseline3/baseline3_2/cifar100")
json_files = sorted(JSON_DIR.glob("*.json"))

summary_list = []
all_results = []

for jf in json_files:
    res = analyze_result(jf)

    if res.get('error'):
        print(f"⚠️ SKIP: {res['file']} - {res['error']}")
        continue

    all_results.append(res)

    summary_list.append({
        'File Name': res['file'],
        'Total': f"{res['total']:,}",
        'Cand Correct': f"{res['c_count']:,}",
        'Cand Acc (%)': f"{res['candidate_acc']:.2f}%",
        'Final Correct': f"{res['f_count']:,}",
        'Final Acc (%)': f"{res['final_acc']:.2f}%",
        'Missed (Cand-Final)': f"{res['missed']:,}",
        'Empty Ans': res['NO/Empty ans']
    })

# --- BẢNG TỔNG HỢP ---
df_summary = pd.DataFrame(summary_list)

print("\n" + "="*50)
print("📊 BẢNG TỔNG HỢP BASELINE ANALYSIS")
print("="*50)
display(df_summary)



📊 BẢNG TỔNG HỢP BASELINE ANALYSIS


,File Name,Total,Cand Correct,Cand Acc (%),Final Correct,Final Acc (%),Missed (Cand-Final),Empty Ans
0,baseline3_2_qwen3_4b_vitl14_cifar100_lbs10.json,"50,000","46,089",92.18%,"39,719",79.44%,"6,370",0
1,baseline3_2_qwen3_4b_vitl14_cifar100_lbs100.json,"50,000","39,355",78.71%,"39,497",78.99%,-142,0
2,baseline3_2_qwen3_4b_vitl14_cifar100_lbs20.json,"50,000","44,614",89.23%,"39,675",79.35%,"4,939",0
3,baseline3_2_qwen3_4b_vitl14_cifar100_lbs5.json,"50,000","46,845",93.69%,"39,488",78.98%,"7,357",0
4,baseline3_2_qwen3_4b_vitl14_cifar100_lbs50.json,"50,000","42,107",84.21%,"39,658",79.32%,"2,449",0
5,baseline3_2_qwen3_8b_vitl14_cifar100_lbs10.json,"50,000","44,409",88.82%,"39,599",79.20%,"4,810",0
6,baseline3_2_qwen3_8b_vitl14_cifar100_lbs100.json,"50,000","39,298",78.60%,"40,683",81.37%,"-1,385",0
7,baseline3_2_qwen3_8b_vitl14_cifar100_lbs20.json,"50,000","42,608",85.22%,"39,507",79.01%,"3,101",0
8,baseline3_2_qwen3_8b_vitl14_cifar100_lbs5.json,"50,000","44,741",89.48%,"39,503",79.01%,"5,238",0
9,baseline3_2_qwen3_8b_vitl14_cifar100_lbs50.json,"50,000","42,826",85.65%,"39,919",79.84%,"2,907",0
